In [1]:
import pandas as pd
import requests
import time
import warnings
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS request.*")

# -----------------------------------
# INPUT FILES
# -----------------------------------
INPUT_FILES = [
    "final_merged_5_geocoded_formatted_address_enriched_with_income_race_poverty.csv",
    "final_merged_24_geocoded_formatted_address_enriched_with_income_race_poverty.csv",
]

# -----------------------------------
# ACS CONFIG
# -----------------------------------
# 2015 -> ACS 2011-2015 5-year -> API year 2015
# 2020 -> ACS 2016-2020 5-year -> API year 2020
# 2025 requested -> use latest available proxy -> ACS 2020-2024 5-year -> API year 2024
ACS_YEARS = {
    "2015": 2015,
    "2020": 2020,
    "2025": 2024,   # proxy for 2025
}

# B25064_001E = Median gross rent
ACS_RENT_VAR = "B25064_001E"


# -----------------------------------
# HELPERS
# -----------------------------------
def tract_code_6_from_row(row):
    """
    Prefer full 11-digit GEOID from 'Census Tract' if available.
    Fallback to 'census_tract_final' / 'census_tract_geocoder'.
    """
    if "Census Tract" in row.index and pd.notna(row["Census Tract"]):
        value = str(row["Census Tract"]).strip().replace(".0", "")
        digits = "".join(ch for ch in value if ch.isdigit())
        if len(digits) >= 11:
            return digits[-6:]

    for col in ["census_tract_final", "census_tract_geocoder", "tract_code_6"]:
        if col in row.index and pd.notna(row[col]):
            value = str(row[col]).strip().replace(".0", "")
            digits = "".join(ch for ch in value if ch.isdigit())
            if len(digits) <= 6 and len(digits) > 0:
                return digits.zfill(6)
            if len(digits) > 6:
                return digits[-6:]

    return None


def fetch_dc_tract_median_rent(api_year):
    """
    Pull DC tract-level median gross rent for one ACS API year.
    """
    url = f"https://api.census.gov/data/{api_year}/acs/acs5"
    params = {
        "get": ACS_RENT_VAR,
        "for": "tract:*",
        "in": "state:11 county:001",  # DC
    }

    r = requests.get(url, params=params, timeout=120, verify=False)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data[1:], columns=data[0])
    df[ACS_RENT_VAR] = pd.to_numeric(df[ACS_RENT_VAR], errors="coerce")

    df["tract_code_6"] = df["tract"].astype(str).str.zfill(6)
    df["tract_geoid"] = df["state"].astype(str) + df["county"].astype(str) + df["tract_code_6"]

    suffix = str(api_year)
    df = df.rename(columns={
        ACS_RENT_VAR: f"median_rent_api_{suffix}"
    })

    return df[["tract_code_6", "tract_geoid", f"median_rent_api_{suffix}"]]


def rename_rent_columns(df):
    """
    Rename API-year rent columns to user-facing names.
    """
    return df.rename(columns={
        "median_rent_api_2015": "median_rent_2015",
        "median_rent_api_2020": "median_rent_2020",
        "median_rent_api_2024": "median_rent_2025_proxy_2024acs",
    })


# -----------------------------------
# FETCH RENT LOOKUP ONCE
# -----------------------------------
lookup_frames = []
for label, api_year in ACS_YEARS.items():
    print(f"Downloading median rent for requested year {label} using API year {api_year}...")
    lookup_frames.append(fetch_dc_tract_median_rent(api_year))
    time.sleep(0.5)

rent_lookup = lookup_frames[0]
for extra in lookup_frames[1:]:
    rent_lookup = rent_lookup.merge(extra, on=["tract_code_6", "tract_geoid"], how="outer")

rent_lookup = rename_rent_columns(rent_lookup)


# -----------------------------------
# APPLY TO EACH FILE
# -----------------------------------
for file_path in INPUT_FILES:
    print(f"\nProcessing {file_path} ...")
    df = pd.read_csv(file_path, low_memory=False)

    # build tract join keys if not already usable
    df["tract_code_6"] = df.apply(tract_code_6_from_row, axis=1)
    df["tract_geoid"] = df["tract_code_6"].apply(
        lambda x: f"11001{x}" if pd.notna(x) else None
    )

    # merge median rent
    df = df.merge(
        rent_lookup,
        on=["tract_code_6", "tract_geoid"],
        how="left"
    )

    output_path = file_path.replace(".csv", "_with_median_rent.csv")
    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print("Rows matched (2015):", df["median_rent_2015"].notna().sum(), "of", len(df))


Processing final_merged_5_geocoded_formatted_address_enriched_with_income_race_poverty.csv ...
Saved: final_merged_5_geocoded_formatted_address_enriched_with_income_race_poverty_with_median_rent.csv
Rows matched (2015): 712 of 893

Processing final_merged_24_geocoded_formatted_address_enriched_with_income_race_poverty.csv ...
Saved: final_merged_24_geocoded_formatted_address_enriched_with_income_race_poverty_with_median_rent.csv
Rows matched (2015): 1619 of 1821
